In [18]:
import kagglehub
import os

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

train_dir = os.path.join(path, 'chest_xray/train')
val_dir = os.path.join(path, 'chest_xray/val')

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.


In [19]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    Input, Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
# Only rescaling
train_gen = ImageDataGenerator(rescale=1./255)
val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

val_data = val_gen.flow_from_directory(
    val_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.


In [20]:
train_gen_basic = ImageDataGenerator(rescale=1./255)
val_gen = ImageDataGenerator(rescale=1./255)

train_data_basic = train_gen_basic.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

val_data = val_gen.flow_from_directory(
    val_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

# WITH augmentation
train_gen_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

train_data_aug = train_gen_aug.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 5216 images belonging to 2 classes.


In [21]:
def build_model():
    model = Sequential([
        Input(shape=(150,150,3)),

        Conv2D(32,(3,3),activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Conv2D(64,(3,3),activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Conv2D(128,(3,3),activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Conv2D(128,(3,3),activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Flatten(),

        Dense(512,activation='relu'),
        Dropout(0.5),

        Dense(1,activation='sigmoid')
    ])
    model.compile(
    optimizer=Adam(learning_rate=0.0001),  # 🔥 tuned LR
    loss='binary_crossentropy',
    metrics=['accuracy']
)
    return model


In [22]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'best_model.h5',
    monitor='val_accuracy',
    save_best_only=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    min_lr=1e-6
)
callbacks = [early_stop, checkpoint, lr_scheduler]

In [23]:
model_basic = build_model()

history_basic = model_basic.fit(
    train_data_basic,
    epochs=10,
    validation_data=val_data,
    callbacks=callbacks
)

Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8934 - loss: 0.3197

163/163 ━━━━━━━━━━━━━━━━━━━━ 461s 3s/step - accuracy: 0.9340 - loss: 0.1921 - val_accuracy: 0.5000 - val_loss: 2.3699 - learning_rate: 1.0000e-04
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 451s 3s/step - accuracy: 0.9734 - loss: 0.0677 - val_accuracy: 0.5000 - val_loss: 5.2503 - learning_rate: 1.0000e-04
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 442s 3s/step - accuracy: 0.9818 - loss: 0.0502 - val_accuracy: 0.5000 - val_loss: 4.9337 - learning_rate: 1.0000e-04
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9859 - loss: 0.0310

163/163 ━━━━━━━━━━━━━━━━━━━━ 456s 3s/step - accuracy: 0.9889 - loss: 0.0271 - val_accuracy: 0.5625 - val_loss: 2.5868 - learning_rate: 3.0000e-05


In [24]:
model_aug = build_model()

history_aug = model_aug.fit(
    train_data_aug,
    epochs=10,
    validation_data=val_data,
    callbacks=callbacks
)

Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 494s 3s/step - accuracy: 0.8779 - loss: 0.3543 - val_accuracy: 0.5000 - val_loss: 5.2908 - learning_rate: 1.0000e-04
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 489s 3s/step - accuracy: 0.9109 - loss: 0.2274 - val_accuracy: 0.5000 - val_loss: 7.7948 - learning_rate: 1.0000e-04
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 486s 3s/step - accuracy: 0.9304 - loss: 0.1834 - val_accuracy: 0.5000 - val_loss: 4.4521 - learning_rate: 3.0000e-05


In [ ]:
plt.plot(history_basic.history['val_accuracy'], label='Without Aug')
plt.plot(history_aug.history['val_accuracy'], label='With Aug')

plt.legend()
plt.title("Augmentation vs No Augmentation")
plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.show()

In [26]:
basic_acc = max(history_basic.history['val_accuracy'])
aug_acc = max(history_aug.history['val_accuracy'])

print("\nFinal Comparison:")
print("Without Augmentation:", basic_acc)
print("With Augmentation:", aug_acc)

if aug_acc > basic_acc:
    print("\nConclusion: Data Augmentation Improved Performance ✅")
else:
    print("\nConclusion: No Significant Improvement ❌")



Final Comparison:
Without Augmentation: 0.5625
With Augmentation: 0.5

Conclusion: No Significant Improvement ❌
